In [2]:
import keras
import numpy as np

solver = keras.models.load_model('sudoku_solver_model_large.keras')
def solve_puzzle(model, puzzle):
    grid = puzzle.copy()
    while (grid[:, :, 0] == 1).any():
        probs = model.predict(grid[np.newaxis], verbose=0)[0]
        best_conf = -1
        best_j = 0
        best_k = 0
        for j in range(9):
            for k in range(9):
                if grid[j, k, 0] == 1 and probs[j, k].max() > best_conf:
                    best_conf = probs[j, k].max()
                    best_j = j
                    best_k = k
        digit = probs[best_j, best_k].argmax() + 1
        grid[best_j, best_k, :] = 0
        grid[best_j, best_k, digit] = 1.0
    return grid.argmax(axis=-1)

def make_input_puzzle():
    one_hot_puzzle = np.zeros((9, 9, 10), dtype=np.float32)
    puzzle = np.array(list(int(x) for x in input('Enter a Sudoku puzzle as a list of 81 digits (0 for empty cells): ')))
    puzzle.reshape(9, 9)
    for j in range(9):
        for k in range(9):
            value_puz = puzzle[j * 9 + k]
            one_hot_puzzle[j, k, value_puz] = 1.0
    return one_hot_puzzle

def calc_performance(y_true, y_pred, labels):
    """Confusion matrix and classification report for a multi-class problem.

    Same helper as Module 3 and the in-class Keras notebook.
    y_true and y_pred are integer labels, not one-hot vectors.
    """
    confusion_matrix_data = confusion_matrix(y_true, y_pred, labels=labels)
    confusion_matrix_display = ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix_data, display_labels=labels
    )
    report_text = classification_report(y_true, y_pred, digits=4)
    report_dict = classification_report(y_true, y_pred, output_dict=True, digits=4)
    return {
        "confusion_matrix_data": confusion_matrix_data,
        "confusion_matrix_display": confusion_matrix_display,
        "report_text": report_text,
        "report_dict": report_dict,
    }

def plot_train_val(history_df, title_prefix=""):
    """Plot training and validation accuracy and loss against epoch.

    history_df must have columns epoch, accuracy, val_accuracy, loss, val_loss.
    Returns the two Axes, (ax_acc, ax_loss).
    """
    fig, ax_acc = plt.subplots(figsize=(8, 4))
    ax_acc.plot(history_df['epoch'], history_df['accuracy'], marker='o', color='blue', label='train')
    ax_acc.plot(history_df['epoch'], history_df['val_accuracy'], marker='o', color='red', label='validation')
    ax_acc.set_xlabel('Epoch')
    ax_acc.set_ylabel('Accuracy')
    ax_acc.set_title(title_prefix + 'Accuracy vs Epoch')
    ax_acc.legend()
    ax_acc.grid(True)
    fig.tight_layout()
    plt.show()

    fig, ax_loss = plt.subplots(figsize=(8, 4))
    ax_loss.plot(history_df['epoch'], history_df['loss'], marker='o', color='blue', label='train')
    ax_loss.plot(history_df['epoch'], history_df['val_loss'], marker='o', color='red', label='validation')
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('Loss')
    ax_loss.set_title(title_prefix + 'Loss vs Epoch')
    ax_loss.legend()
    ax_loss.grid(True)
    fig.tight_layout()
    plt.show()
    return ax_acc, ax_loss

In [2]:
puzzle = make_input_puzzle()
solution = solve_puzzle(solver, puzzle)
print("Solved Sudoku Puzzle:")
print(solution)
#519030007000050010300000000001005470800374102467129500000200005040501026652740090

Solved Sudoku Puzzle:
[[2 1 4 5 7 6 9 8 3]
 [9 8 7 4 3 1 5 6 2]
 [6 5 3 8 9 2 4 7 1]
 [3 7 1 2 4 9 6 5 8]
 [5 9 8 6 1 7 3 2 4]
 [4 2 6 3 5 8 1 9 7]
 [8 4 5 7 6 3 2 1 9]
 [1 6 2 9 8 4 7 3 5]
 [7 3 9 1 2 5 8 4 6]]
